In [63]:
"""
HomeGuard Security System Simulator
Author: Javier Portero Aylagas
Description: A smart home monitoring system that processes sensor readings
             and triggers alerts for security, safety, and comfort issues.
"""

import time
import random
from datetime import datetime
 
# System configuration
HOME_MODES = ["HOME", "AWAY", "SLEEP"]
ALERT_SEVERITIES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
 
# Current system state
current_mode = "AWAY"

# Background Story
HomeGuard's client, the Peterson family, wants to monitor their vacation home while they're away. They've installed:

- Motion sensors in each room to detect intruders
- Temperature sensors to prevent frozen pipes or fire hazards
- Door sensors to track entry points
- Smoke detectors for fire safety

Your simulator needs to:

- Generate realistic sensor readings
- Detect abnormal conditions
- Trigger appropriate alerts
- Log all events for review

# System Requirements
The simulator must handle these scenarios:

Security Alerts:

- Motion detected when house is in "away" mode
- Door opened when house is in "away" mode
- Multiple sensors triggered simultaneously (possible break-in)

Safety Alerts:

- Temperature below 35°F (frozen pipe risk)
- Temperature above 95°F (equipment failure)
- Smoke detected (fire risk)

Comfort Notifications:

- Temperature outside comfort range (65-75°F) when home is in "home" mode
- Unusual patterns (like a door left open for >5 minutes)

## Step 2: Creating Data Structures
**Objective:** Define the data structures (dictionaries) that will represent sensors, alerts, and system state.

In [64]:
def create_sensor(sensor_id, location, sensor_type, threshold=None):
    """
    Creates a sensor data structure.
    
    Parameters:
    - sensor_id: Unique identifier for the sensor
    - location: Where the sensor is located (e.g., "Living Room")
    - sensor_type: Type of sensor ("motion", "temperature", "door", "smoke")
    - threshold: Optional threshold value for the sensor
    
    Returns:
    - A dictionary representing the sensor
    """
    sensor = {
        "id" : sensor_id,
        "location": location, 
        "type": sensor_type,
        "threshold": threshold 
    }
    return sensor
 
def create_alert(severity, message, sensor_id, timestamp):
    """
    Creates an alert data structure.
    
    Parameters:
    - severity: Alert severity level (LOW, MEDIUM, HIGH, CRITICAL)
    - message: Description of the alert
    - sensor_id: ID of the sensor that triggered the alert
    - timestamp: When the alert was triggered
    
    Returns:
    - A dictionary representing the alert
    """
    alert = {
        "severity": severity,
        "message": message,
        "sensor_id": sensor_id, 
        "timestamp": timestamp
        }
    return alert
 
# Initialize sensors for the Peterson home
sensors = [
    create_sensor("MOTION_001", "Living Room", "motion"), #motion for the living room 
    create_sensor("TEMP_001", "Kitchen", "temperature"), #temperature for the kitchen with threshold (ToDo: add threshold value?)
    create_sensor("DOOR_001", "Front Door", "door"), #door for the front door
    create_sensor("SMOKE_001", "Bedroom", "smoke")  #smoke for the bedroom 
]

In [65]:
print(f"Initialized {len(sensors)} sensors")
for sensor in sensors:
    print(f"  - {sensor['id']}: {sensor['location']} ({sensor['type']})")

Initialized 4 sensors
  - MOTION_001: Living Room (motion)
  - TEMP_001: Kitchen (temperature)
  - DOOR_001: Front Door (door)
  - SMOKE_001: Bedroom (smoke)


## Step 3: Implementing If/Else Logic
**Objective:** Write conditional statements to check sensor thresholds and system modes.

In [ ]:
def is_abnormal_reading(sensor, reading_value):
    """
    Checks if a sensor reading is abnormal based on sensor type and thresholds.
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The current reading from the sensor
    
    Returns:
    - True if the reading is abnormal, False otherwise
    """
    sensor_type = sensor["type"]
    
    # Temperature sensors: check if below 35°F or above 95°F
    if sensor_type == "temperature":
        return reading_value < 35 or reading_value > 95
    
    # Motion sensors: check if motion is detected (reading_value == True)
    elif sensor_type == "motion":
        return reading_value is True
    
    # Door sensors: check if door is open (reading_value == "OPEN")
    # note: door opening time 5 minutes checked in the simulation loop
    elif sensor_type == "door":
        return reading_value == "OPEN"
    
    # Smoke sensors: check if smoke is detected (reading_value == "DETECTED")
    elif sensor_type == "smoke":
        return reading_value == "DETECTED"
    
    else:
        return False

def is_outside_comfort_reading(sensor, reading_value):
    """
    Checks if a sensor reading is outside the comfort range.

    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The current reading from the sensor

    Returns:
    - True if the reading is outside the comfort range, False otherwise
    """
    
    # Temperature sensors: check if below 65°F or above 75°F
    sensor_type = sensor["type"]

    if sensor_type == "temperature":
        return reading_value < 65 or reading_value > 75
    else:
        return False
    


def should_trigger_security_alert(sensor, reading_value, system_mode):
    """
    Determines if a security alert should be triggered.
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The current reading from the sensor
    - system_mode: Current system mode (HOME, AWAY, SLEEP)
    
    Returns:
    - True if a security alert should be triggered, False otherwise
    """
    if system_mode == "AWAY" and sensor["type"] in ["motion", "door"]:
        return is_abnormal_reading(sensor, reading_value)   
    else:
        return False    

In [67]:
test_sensor = create_sensor("TEMP_TEST", "Test Room", "temperature", threshold=35)
print(f"34°F is abnormal: {is_abnormal_reading(test_sensor, 34)}")  # Should be True
print(f"68°F is abnormal: {is_abnormal_reading(test_sensor, 68)}")  # Should be False
 
# Test security alert
motion_sensor = create_sensor("MOTION_TEST", "Test Room", "motion")
print(f"Motion in AWAY mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'AWAY')}")  # Should be True
print(f"Motion in HOME mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'HOME')}")  # Should be False

34°F is abnormal: True
68°F is abnormal: False
Motion in AWAY mode triggers alert: True
Motion in HOME mode triggers alert: False


## Step 4: Building Functions
**Objective:** Create functions to generate sensor readings, process them, trigger alerts, and log events.

In [ ]:
def generate_reading(sensor):
    """
    Generates a realistic reading for a sensor by simulating real sensor data.
    
    Parameters:
    - sensor: Sensor dictionary
    
    Returns:
    - A realistic reading value based on sensor type
    """    
    sensor_type = sensor["type"]
    # Temperature: random value between 30-100°F
    if sensor_type == "temperature":
        return random.randint(30, 100)
    
    # Motion: random boolean (True/False)
    elif sensor_type == "motion":
        return random.choice([True, False])   
    # Door: random choice between "OPEN" and "CLOSED"
    elif sensor_type == "door":
        return random.choice(["OPEN", "CLOSED"])
    
    # Smoke: random choice between "CLEAR" and "DETECTED"
    elif sensor_type == "smoke":
        return random.choice(["CLEAR", "DETECTED"])
    
    else:
        return None
 
def process_reading(sensor, reading_value, system_mode):
    """
    Processes a sensor reading and determines if alerts are needed.
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The reading from the sensor
    - system_mode: Current system mode
    
    Returns:
    - A list of alerts (empty if no alerts needed)
    """
    # Severity policy:
    # CRITICAL -> intrusion, fire, or possible break-in
    # HIGH     -> overheating risk
    # MEDIUM   -> freezing risk
    # LOW      -> comfort issue or door left open for 5 minutes
    #
    # Note:
    # POSSIBLE_BREAK_IN and DOOR_LEFT_OPEN are handled in run_simulation()
    # because they depend on multiple sensors or multiple time steps.
    
    alerts = []
    timestamp = datetime.now().strftime("%H:%M:%S")
    
    sensor_id = sensor["id"]
    sensor_type = sensor["type"]

    # 1. Check for security alerts (motion/door in AWAY mode)
    if should_trigger_security_alert(sensor, reading_value, system_mode):
        alerts.append(create_alert("CRITICAL", "SECURITY_ALERT", sensor_id, timestamp))

    # 2. Check for safety alerts (temperature extremes, smoke)
    if sensor_type == "temperature":
        if reading_value < 35:
            alerts.append(create_alert("MEDIUM", "TEMP_LOW", sensor_id, timestamp))
        elif reading_value > 95:
            alerts.append(create_alert("HIGH", "TEMP_HIGH", sensor_id, timestamp))

    elif sensor_type == "smoke":
        if reading_value == "DETECTED":
            alerts.append(create_alert("CRITICAL", "SMOKE_DETECTED", sensor_id, timestamp))

    # 3. Check for comfort notifications (temperature out of range in HOME mode)
    if system_mode == "HOME" and sensor_type == "temperature":
        if is_outside_comfort_reading(sensor, reading_value):
            alerts.append(create_alert("LOW", "TEMP_COMFORT", sensor_id, timestamp))

    return alerts
 
def trigger_alert(alert):
    """
    Displays an alert to the user.
    
    Parameters:
    - alert: Alert dictionary
    """
    severity_symbol = {
        "LOW": "ℹ️",
        "MEDIUM": "⚠️",
        "HIGH": "🚨",
        "CRITICAL": "🔥"
    }
    
    symbol = severity_symbol.get(alert["severity"])
    print(f"[ALERT!] {symbol} {alert['severity']}: {alert['message']}")
 
def log_event(message, timestamp=None):
    """
    Logs an event to the console.
    
    Parameters:
    - message: The message to log
    - timestamp: Optional timestamp (uses current time if not provided)
    """
    if timestamp is None:
        timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")

In [69]:
# Test reading generation
test_sensor = sensors[0]  # Motion sensor
reading = generate_reading(test_sensor)
print(f"Generated reading for {test_sensor['location']}: {reading}")
 
# Test processing
alerts = process_reading(test_sensor, True, "AWAY")
print(alerts)
if alerts:
    trigger_alert(alerts[0])

Generated reading for Living Room: True
[{'severity': 'CRITICAL', 'message': 'SECURITY_ALERT', 'sensor_id': 'MOTION_001', 'timestamp': '14:34:54'}]
[ALERT!] 🔥 CRITICAL: SECURITY_ALERT


## Step 5: Creating Classes
**Objective:** Convert the sensor dictionary into a Sensor class with methods.

In [ ]:
class Sensor:
    """
    Represents a sensor in the HomeGuard system.
    """
    
    def __init__(self, sensor_id, location, sensor_type, threshold=None):
        """
        Initializes a new sensor.
        
        Parameters:
        - sensor_id: Unique identifier for the sensor
        - location: Where the sensor is located
        - sensor_type: Type of sensor ("motion", "temperature", "door", "smoke")
        - threshold: Optional threshold value for the sensor
        """
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.threshold = threshold
        self.current_value = None
        self.open_minutes = 0 # Track how long door stays open (in simulation "minutes")
    
    def read(self):
        """
        Generates and stores a new reading for this sensor.
        
        Returns:
        - The reading value
        """
        # Generate a reading and store it in self.current_value
        self.current_value = None
        
        sensor_dict = {
            "id": self.id,
            "location": self.location,
            "type": self.type,
            "threshold": self.threshold
        }

        self.current_value = generate_reading(sensor_dict)
        return self.current_value
 
        
    def isAbnormal(self):
        """
        Checks if the current reading is abnormal.
        
        Returns:
        - True if the reading is abnormal, False otherwise
        """
        sensor_dict = {
            "id": self.id,
            "location": self.location,
            "type": self.type,
            "threshold": self.threshold
        }
        return is_abnormal_reading(sensor_dict, self.current_value)
        
  
    def reset(self):
        """
        Resets the sensor's current reading to None.
        """
        self.current_value = None
    
    def __str__(self):
        """
        Returns a string representation of the sensor.
        """
        status = "No reading" if self.current_value is None else str(self.current_value)
        return f"{self.id} ({self.location}): {status}"
 
# Create sensor objects using the class
sensor_objects = [
    Sensor("MOTION_001", "Living Room", "motion"),
    Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
    Sensor("DOOR_001", "Front Door", "door"),
    Sensor("SMOKE_001", "Bedroom", "smoke")
]

In [71]:
# Create and test a sensor
test_sensor = Sensor("TEST_001", "Test Room", "temperature", threshold=35)
test_sensor.read()
print(f"Sensor reading: {test_sensor.current_value}")
print(f"Is abnormal: {test_sensor.isAbnormal()}")
print(f"Sensor info: {test_sensor}")

Sensor reading: 62
Is abnormal: False
Sensor info: TEST_001 (Test Room): 62


## Step 6: Integrating Everything
**Objective:** Combine all components into a complete simulator that runs continuously.

In [74]:
def run_simulation(duration_minutes=5, system_mode="AWAY"):
    """
    Runs the HomeGuard security system simulation.
    
    Parameters:
    - duration_minutes: How long to run the simulation (default: 5 minutes)
    - system_mode: System mode (HOME, AWAY, SLEEP)
    """
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")
    
    sensors = [
        Sensor("MOTION_001", "Living Room", "motion"),
        Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
        Sensor("DOOR_001", "Front Door", "door"),
        Sensor("SMOKE_001", "Bedroom", "smoke")
    ]
    
    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")

        # Count how many security-related sensors were triggered
        # during this simulation minute (used for possible break-in detection)
        security_triggers = 0
        
        for sensor in sensors:
            reading = sensor.read()
            
            if sensor.type == "temperature":
                if reading < 35:
                    status = "Frozen Pipe Risk"
                elif reading < 65:
                    status = "Below Comfort"
                elif reading <= 75:
                    status = "Comfort Range"
                elif reading <= 95:
                    status = "Above Comfort"
                else:
                    status = "Equipment Failure"

                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")

            elif sensor.type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")

            elif sensor.type == "door":
                print(f"[READING] {sensor.location}: {reading}")

                # Track how long the door has remained open
                if reading == "OPEN":
                    sensor.open_minutes += 1
                else:
                    sensor.open_minutes = 0

            elif sensor.type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")
            
            sensor_dict = {
                "id": sensor.id,
                "location": sensor.location,
                "type": sensor.type,
                "threshold": sensor.threshold
            }

            # Count individual security triggers for combined break-in detection
            if should_trigger_security_alert(sensor_dict, reading, system_mode):
                security_triggers += 1

            alerts = process_reading(sensor_dict, reading, system_mode)
            
            for alert in alerts:
                trigger_alert(alert)

                if alert["severity"] in ["MEDIUM", "HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")

            # Trigger a low-priority alert if the door stays open for 5 simulation minutes
            if sensor.type == "door" and sensor.open_minutes == 5:
                door_alert = create_alert("LOW", "DOOR_LEFT_OPEN", sensor.id, current_time)
                trigger_alert(door_alert)

        # Trigger a combined break-in alert if multiple security sensors
        # were activated in the same simulation minute
        if system_mode == "AWAY" and security_triggers >= 2:
            break_in_alert = create_alert("CRITICAL", "POSSIBLE_BREAK_IN", "SYSTEM", current_time)
            trigger_alert(break_in_alert)
            log_event("Multiple security sensors triggered: possible break-in detected!")

        time.sleep(0.5)
    
    print("\n" + "=" * 50)
    print("Simulation complete!")
    print("=" * 50)


# Main execution
if __name__ == "__main__":
    # Run the simulation in away mode
    run_simulation(duration_minutes=10, system_mode="AWAY")

    # Run the simulation in home mode
    run_simulation(duration_minutes=10, system_mode="HOME")

    # Run the simulation in sleep mode
    run_simulation(duration_minutes=10, system_mode="SLEEP")

=== HomeGuard Security System ===
Mode: AWAY


Time: 15:02:08
[READING] Living Room Motion: DETECTED
[ALERT!] 🔥 CRITICAL: SECURITY_ALERT
[LOG] [15:02:08] Sending notification to homeowner...
[READING] Kitchen Temperature: 90°F (Above Comfort)
[READING] Front Door: OPEN
[ALERT!] 🔥 CRITICAL: SECURITY_ALERT
[LOG] [15:02:08] Sending notification to homeowner...
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: SMOKE_DETECTED
[LOG] [15:02:08] Sending notification to homeowner...
[ALERT!] 🔥 CRITICAL: POSSIBLE_BREAK_IN
[LOG] [15:02:08] Multiple security sensors triggered: possible break-in detected!

Time: 15:02:08
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 53°F (Below Comfort)
[READING] Front Door: OPEN
[ALERT!] 🔥 CRITICAL: SECURITY_ALERT
[LOG] [15:02:08] Sending notification to homeowner...
[READING] Bedroom Smoke: CLEAR

Time: 15:02:09
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 98°F (Equipment Failure)
[ALERT!] 🚨 HIGH: TEMP_H